In [15]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [16]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

### Images

In [17]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [18]:
# TODO: Read image data, feed into Claude
with open("./images/prop1.png", "rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = [] 

add_user_message(messages, [
    # Image Block
    {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": "image/png",
            "data": image_bytes,
        }
    },
    # Text Block
    {
        "type": "text",
        "text": prompt # "What do you see in this image?"
    }
])

In [19]:
response = chat(messages)     #, prompt)
response 

Message(id='msg_01BYoTQm6UNpg7EasXgaJknb', container=None, content=[TextBlock(citations=None, text='# Satellite Image Analysis - Fire Risk Assessment\n\n**1. Residence Identification:**\nThe primary residence is located in the center-left portion of the image, identifiable as a tan/beige rectangular roofed structure with regular geometry, surrounded by dense vegetation on all sides.\n\n**2. Tree Overhang Analysis:**\nMultiple trees have canopies that extend directly over the roof of the residence, with approximately 50-75% of the roof area covered by overhanging branches, particularly dense on the northern and eastern portions of the structure.\n\n**3. Fire Risk Assessment:**\nThe overhanging trees create significant wildfire vulnerability by providing direct pathways for embers to reach the roof, with dense canopy coverage creating multiple ember catch points and a continuous fuel bridge from surrounding vegetation to the structure.\n\n**4. Defensible Space Identification:**\nThe prop

In [20]:
len(response.content)

1

In [21]:
print(response.content[0].text)

# Satellite Image Analysis - Fire Risk Assessment

**1. Residence Identification:**
The primary residence is located in the center-left portion of the image, identifiable as a tan/beige rectangular roofed structure with regular geometry, surrounded by dense vegetation on all sides.

**2. Tree Overhang Analysis:**
Multiple trees have canopies that extend directly over the roof of the residence, with approximately 50-75% of the roof area covered by overhanging branches, particularly dense on the northern and eastern portions of the structure.

**3. Fire Risk Assessment:**
The overhanging trees create significant wildfire vulnerability by providing direct pathways for embers to reach the roof, with dense canopy coverage creating multiple ember catch points and a continuous fuel bridge from surrounding vegetation to the structure.

**4. Defensible Space Identification:**
The property shows minimal defensible space, with trees forming a nearly continuous canopy that connects directly to and

### PDF

In [ ]:
with open("earth.pdf", "rb") as f:
    pdf_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

pdf_bytes

'...'

In [23]:
messages = []

add_user_message(
    messages,
    [
        {
            "type": "document",
            "source": {
                "type": "base64",
                "media_type": "application/pdf",
                "data": pdf_bytes,
            },
        },
        {"type": "text", "text": "Summarize the document in one sentence"},
    ],
)

response = chat(messages)
response

Message(id='msg_01U6o7cLnHs3XhJYe2nNZtow', container=None, content=[TextBlock(citations=None, text='This Wikipedia article provides comprehensive information about Earth, including its physical characteristics, formation about 4.5 billion years ago, its status as the only known planet to harbor life, its composition with 70.8% ocean coverage, and details about its atmosphere, orbital mechanics, and geological history.', type='text')], model='claude-sonnet-4-5-20250929', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=9625, output_tokens=63, server_tool_use=None, service_tier='standard'))

In [26]:
len(response.content), response.content[0].text

(1,
 'This Wikipedia article provides comprehensive information about Earth, including its physical characteristics, formation about 4.5 billion years ago, its status as the only known planet to harbor life, its composition with 70.8% ocean coverage, and details about its atmosphere, orbital mechanics, and geological history.')

### Citations

In [27]:
with open("earth.pdf", "rb") as f:
    pdf_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

pdf_bytes

'...'

In [28]:
messages = []

add_user_message(
    messages,
    [
        {
            "type": "document",
            "source": {
                "type": "base64",
                "media_type": "application/pdf",
                "data": pdf_bytes,
            },
            "title": "earth.pdf",
            "citations": {
                "enabled": True, 
            }
        },
        {"type": "text", "text": "How earth atmosphere and oceans are formed?"},
    ],
)

response = chat(messages)
response

Message(id='msg_01UQkpCdqyaYkzRScrebXDXT', container=None, content=[TextBlock(citations=[CitationPageLocation(cited_text="[42]\r\nEarth's atmosphere and oceans were formed by volcanic activity and outgassing.\r\n", document_index=0, document_title='earth.pdf', end_page_number=5, file_id=None, start_page_number=4, type='page_location')], text="Earth's atmosphere and oceans were formed by volcanic activity and outgassing.", type='text'), TextBlock(citations=None, text=' ', type='text'), TextBlock(citations=[CitationPageLocation(cited_text='[43] Water vapor from\r\nthese sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets,\r\nand comets.\r\n', document_index=0, document_title='earth.pdf', end_page_number=5, file_id=None, start_page_number=4, type='page_location')], text='Water vapor from these sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets, and comets.', type='text')], model='claude-sonnet-4-5-20250929', ro

In [31]:
len(response.content)

3

In [42]:
for index, block in enumerate(response.content):
    print(f"Block index: {index} - Block type: {block.type}")
    print(f"Citations: {len(block.citations) if block.citations else 0} and {block.citations}")
    print(f"Text: {block.text}\n") 

Block index: 0 - Block type: text
Citations: 1 and [CitationPageLocation(cited_text="[42]\r\nEarth's atmosphere and oceans were formed by volcanic activity and outgassing.\r\n", document_index=0, document_title='earth.pdf', end_page_number=5, file_id=None, start_page_number=4, type='page_location')]
Text: Earth's atmosphere and oceans were formed by volcanic activity and outgassing.

Block index: 1 - Block type: text
Citations: 0 and None
Text:  

Block index: 2 - Block type: text
Citations: 1 and [CitationPageLocation(cited_text='[43] Water vapor from\r\nthese sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets,\r\nand comets.\r\n', document_index=0, document_title='earth.pdf', end_page_number=5, file_id=None, start_page_number=4, type='page_location')]
Text: Water vapor from these sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets, and comets.



#### Cited with plain text

In [44]:
article_text = """
    Earth is the third planet from the Sun and the only astronomical object known to harbor life. This is made possible by Earth being an ocean world, the only one in the Solar System sustaining liquid surface water. Almost all of Earth's water is contained in its global ocean, covering 70.8% of Earth's crust. The remaining 29.2% of Earth's crust is land, most of which is located in the form of continental landmasses within Earth's land hemisphere. Most of Earth's land is at least somewhat humid and covered by vegetation, while large ice sheets at Earth's polar deserts retain more water than Earth's groundwater, lakes, rivers, and atmospheric water combined. Earth's crust consists of slowly moving tectonic plates, which interact to produce mountain ranges, volcanoes, and earthquakes. Earth has a liquid outer core that generates a magnetosphere capable of deflecting most of the destructive solar winds and cosmic radiation.

    Earth has a dynamic atmosphere, which sustains Earth's surface conditions and protects it from most meteoroids and UV-light at entry. It is composed primarily of nitrogen and oxygen. Water vapor is widely present in the atmosphere, forming clouds that cover most of the planet. The water vapor acts as a greenhouse gas and, together with other greenhouse gases in the atmosphere, particularly carbon dioxide (CO2), creates the conditions for both liquid surface water and water vapor to persist via the capturing of energy from the Sun's light. This process maintains the current average surface temperature of 14.76 °C (58.57 °F), at which water is liquid under normal atmospheric pressure. Differences in the amount of captured energy between geographic regions (as with the equatorial region receiving more sunlight than the polar regions) drive atmospheric and ocean currents, producing a global climate system with different climate regions, and a range of weather phenomena such as precipitation, allowing components such as carbon and nitrogen to cycle.

    Earth is rounded into an ellipsoid with a circumference of about 40,000 kilometers (24,900 miles). It is the densest planet in the Solar System. Of the four rocky planets, it is the largest and most massive. Earth is about eight light-minutes (1 AU) away from the Sun and orbits it, taking a year (about 365.25 days) to complete one revolution. Earth rotates around its own axis in slightly less than a day (in about 23 hours and 56 minutes). Earth's axis of rotation is tilted with respect to the perpendicular to its orbital plane around the Sun, producing seasons. Earth is orbited by one permanent natural satellite, the Moon, which orbits Earth at 384,400 km (238,855 mi)—1.28 light seconds—and is roughly a quarter as wide as Earth. The Moon's gravity helps stabilize Earth's axis, causes tides and gradually slows Earth's rotation. Likewise, Earth's gravitational pull has already made the Moon's rotation tidally locked, keeping the same near side facing Earth.
"""

In [45]:
messages = []

add_user_message(
    messages,
    [
        {
            "type": "document",
            "source": {
                "type": "text",
                "media_type": "text/plain",
                "data": article_text,
            },
            "title": "Earth Article",
            "citations": {
                "enabled": True, 
            }
        },
        {"type": "text", "text": "How earth atmosphere and oceans are formed?"},
    ],
)

response = chat(messages) 
response

Message(id='msg_01KLWpuzpGaQkV1PYFSLyEEn', container=None, content=[TextBlock(citations=None, text="I appreciate your question, but the document provided does not contain information about how Earth's atmosphere and oceans were formed. \n\nThe document describes the current state of Earth's atmosphere and oceans - for example, it mentions that ", type='text'), TextBlock(citations=[CitationCharLocation(cited_text="Earth has a dynamic atmosphere, which sustains Earth's surface conditions and protects it from most meteoroids and UV-light at entry. It is composed primarily of nitrogen and oxygen. ", document_index=0, document_title='Earth Article', end_char_index=1125, file_id=None, start_char_index=942, type='char_location')], text='Earth has a dynamic atmosphere composed primarily of nitrogen and oxygen', type='text'), TextBlock(citations=None, text=' and that ', type='text'), TextBlock(citations=[CitationCharLocation(cited_text="This is made possible by Earth being an ocean world, the o

In [46]:
for index, block in enumerate(response.content):
    print(f"Block index: {index} - Block type: {block.type}")
    print(f"Citations: {len(block.citations) if block.citations else 0} and {block.citations}")
    print(f"Text: {block.text}\n") 

Block index: 0 - Block type: text
Citations: 0 and None
Text: I appreciate your question, but the document provided does not contain information about how Earth's atmosphere and oceans were formed. 

The document describes the current state of Earth's atmosphere and oceans - for example, it mentions that 

Block index: 1 - Block type: text
Citations: 1 and [CitationCharLocation(cited_text="Earth has a dynamic atmosphere, which sustains Earth's surface conditions and protects it from most meteoroids and UV-light at entry. It is composed primarily of nitrogen and oxygen. ", document_index=0, document_title='Earth Article', end_char_index=1125, file_id=None, start_char_index=942, type='char_location')]
Text: Earth has a dynamic atmosphere composed primarily of nitrogen and oxygen

Block index: 2 - Block type: text
Citations: 0 and None
Text:  and that 

Block index: 3 - Block type: text
Citations: 1 and [CitationCharLocation(cited_text="This is made possible by Earth being an ocean world,